# SPINE-GPE v7 — PNAD COVID Certifier v1.0.2

Hardening da série maio–novembro/2020:

- corrige `C007` (`07`, `7.0`, `7` → `7`);
- recalcula `pandemic_delivery_self_employed`;
- audita cobertura e missingness de `C014`;
- cria outputs imutáveis por Run ID;
- promove o alias `latest` apenas após aprovação dos gates críticos.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_CANDIDATES = [
    Path("/content/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.py"),
    ROOT / "scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.py",
]
REQ_CANDIDATES = [
    Path("/content/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.txt"),
    ROOT / "scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.txt",
]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), SCRIPT_CANDIDATES[0])
REQ = next((p for p in REQ_CANDIDATES if p.exists()), REQ_CANDIDATES[0])

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("REQ:", REQ, REQ.exists())
if not SCRIPT.exists():
    raise FileNotFoundError(f"Script não encontrado: {SCRIPT_CANDIDATES}")
if not REQ.exists():
    raise FileNotFoundError(f"Requirements não encontrado: {REQ_CANDIDATES}")


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "-r", str(REQ)],
    check=True,
)
subprocess.run([sys.executable, "-m", "py_compile", str(SCRIPT)], check=True)
print("py_compile: OK")


## 1. Auditoria da série completa


In [ ]:
audit = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--root", str(ROOT),
        "--mode", "audit",
        "--months", "all",
        "--download",
        "--strict",
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print("Audit exit code:", audit.returncode)
if audit.returncode != 0:
    raise RuntimeError("A auditoria falhou; revise STDOUT/STDERR antes da certificação.")


## 2. Certificação endurecida maio–novembro/2020


In [ ]:
cert = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--root", str(ROOT),
        "--mode", "certify",
        "--months", "all",
        "--download",
        "--chunk-rows", "50000",
        "--strict",
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(cert.stdout)
print(cert.stderr)
print("Certification exit code:", cert.returncode)


## 3. Lock, outputs imutáveis e hashes


In [ ]:
LOCK = ROOT / "00_admin/PNAD_COVID_CERTIFICATION_LOCK.json"
if not LOCK.exists():
    raise RuntimeError("O lock não foi criado. Revise a célula de certificação.")

lock = json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock, ensure_ascii=False, indent=2))

if lock.get("status") not in {"CERTIFIED", "CORE_CERTIFIED"}:
    raise RuntimeError(
        f"Certificação não liberada: {lock.get('status')}. "
        f"Falhas: {lock.get('critical_failures', [])}"
    )

immutable_output = Path(lock["output"])
latest_alias = Path(lock["output_latest"])
assert immutable_output.exists(), immutable_output
assert latest_alias.exists(), latest_alias
assert lock["run_id"] in immutable_output.name


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(2**20), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(immutable_output) == sha256(latest_alias)
assert sha256(immutable_output) == lock["artifact_hashes"]["output"]
print("STATUS:", lock["status"])
print("OUTPUT IMUTÁVEL:", immutable_output)
print("ALIAS LATEST:", latest_alias)
print("SHA-256:", sha256(immutable_output))


## 4. Verificação independente de conta-própria e C014


In [ ]:
import pandas as pd

cols = [
    "reference_month",
    "C007",
    "C014",
    "pandemic_delivery_observed",
    "pandemic_delivery_self_employed",
    "social_security_response_valid",
    "social_security_contributor",
]
df = pd.read_parquet(immutable_output, columns=cols)
delivery = df[df["pandemic_delivery_observed"].fillna(False)].copy()

summary = (
    delivery.groupby("reference_month", observed=True)
    .agg(
        n_delivery=("pandemic_delivery_observed", "size"),
        n_self_employed=("pandemic_delivery_self_employed", lambda s: int(s.fillna(False).sum())),
        n_C014_valid=("social_security_response_valid", lambda s: int(s.fillna(False).sum())),
        n_C014_missing=("social_security_response_valid", lambda s: int((~s.fillna(False)).sum())),
    )
    .reset_index()
)
summary["self_employed_percent_unweighted"] = (
    summary["n_self_employed"] / summary["n_delivery"] * 100
)
summary["C014_coverage_percent_unweighted"] = (
    summary["n_C014_valid"] / summary["n_delivery"] * 100
)
print(summary.to_string(index=False))

assert set(delivery["C007"].dropna().astype(str).unique())
assert set(delivery["C014"].dropna().astype(str).unique()) <= {"1", "2"}
assert (summary["n_self_employed"] > 0).all(), "Conta-própria permaneceu zerada."

for row in summary.to_dict("records"):
    audit_row = lock["coverage_audits"][str(int(row["reference_month"]))]
    assert audit_row["n_delivery"] == row["n_delivery"]
    assert audit_row["n_self_employed"] == row["n_self_employed"]
    assert audit_row["n_C014_valid"] == row["n_C014_valid"]
    assert audit_row["n_C014_missing"] == row["n_C014_missing"]

print("Hardening C007/C014: OK")


## 5. Abrir o relatório canônico


In [ ]:
report_path = Path(lock["report"])
print("Relatório imutável:", report_path)
print(report_path.read_text(encoding="utf-8")[:12000])
